In [7]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectFromModel
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix

# -------------------------
# 0) Daten laden
# -------------------------
df = pd.read_csv("survey_results_cleaned_final.csv")
print("Loaded:", df.shape)

# Bool-Spalten ggf. in int umwandeln
bool_cols = df.select_dtypes(include=["bool"]).columns
for col in bool_cols:
    df[col] = df[col].astype(int)

target_col = "Employment"

# Target muss vorhanden sein
df = df.dropna(subset=[target_col]).copy()

# ✅ NEU: seltene/unnütze Klasse droppen (verhindert 1-sample Klassen im Test)
df = df[df[target_col] != "i prefer not to say"].copy()

# Optional: "Other" extrem selten -> droppen (bei dir waren es 2)
df = df[df[target_col] != "Other"].copy()

print("\nTarget distribution:")
print(df[target_col].value_counts())

# -------------------------
# 1) Feature-Spalten bestimmen
# -------------------------
text_cols = df.select_dtypes(include=["object"]).columns.tolist()
text_cols = [c for c in text_cols if c not in {"cluster", target_col}]  # target nicht in Text
df["__text__"] = df[text_cols].fillna("").agg(" ".join, axis=1)

num_cols = df.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()
num_cols = [c for c in num_cols if c not in {target_col, "cluster"}]

# Optional: ID raus, falls vorhanden
for maybe_id in ["ResponseId"]:
    if maybe_id in num_cols:
        num_cols.remove(maybe_id)

print("\nTextspalten:", len(text_cols))
print("Numerische Spalten:", len(num_cols))

# ✅ NEU: Numerische NaNs droppen (oder imputer verwenden)
df = df.dropna(subset=num_cols).copy()

X = df[["__text__"] + num_cols].copy()
y = df[target_col].astype(str).copy()

print("\nX shape:", X.shape)
print("y shape:", y.shape)

# -------------------------
# 2) Train/Test Split
# -------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTrain size:", len(X_train))
print("Test size:", len(X_test))
print("Train dist:\n", y_train.value_counts(normalize=True))
print("Test dist:\n", y_test.value_counts(normalize=True))

# -------------------------
# 3) Preprocessing
# -------------------------
preprocessor = ColumnTransformer(
    transformers=[
        ("text", TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 2),
            min_df=5,            # ✅ NEU (vorher 2)
            max_df=0.9,
            max_features=20000,  # ✅ NEU
            sublinear_tf=True    # ✅ NEU
        ), "__text__"),
        ("num", StandardScaler(), num_cols)
    ],
    remainder="drop"
)

# -------------------------
# 4) Pipeline
# -------------------------
pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("feature_selection", SelectFromModel(
        LinearSVC(penalty="l1", dual=False, C=0.5, max_iter=20000)
    )),
    ("classifier", LinearSVC(max_iter=20000))
])

# -------------------------
# 5) GridSearch
# -------------------------
parameters = {
    "preprocessing__text__ngram_range": [(1, 1), (1, 2)],
    "preprocessing__text__min_df": [5, 10],      # ✅ NEU: stabilere Varianten
    "preprocessing__text__max_df": [0.9],

    "classifier__C": [0.5, 1.0, 2.0],
    "classifier__class_weight": [None, "balanced"],
}

grid = GridSearchCV(
    pipeline,
    param_grid=parameters,
    scoring="f1_macro",
    verbose=2,
    cv=3,
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("\nBeste Performance (CV, f1_macro):", grid.best_score_)
print("Beste Parameter:\n", grid.best_params_)

# -------------------------
# 6) Evaluation
# -------------------------
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

print("\nClassification Report (Test):")
print(classification_report(y_test, y_pred, zero_division=0))

print("Confusion Matrix (rows=true, cols=pred):")
labels_sorted = sorted(y.unique())
print(confusion_matrix(y_test, y_pred, labels=labels_sorted))

print("\nClassification Report (Train):")
y_pred_train = best_model.predict(X_train)
print(classification_report(y_train, y_pred_train, zero_division=0))


Loaded: (19826, 38)

Target distribution:
Employment
employed                                                16316
independent contractor, freelancer, or self-employed     2483
student                                                   659
not employed                                              352
Name: count, dtype: int64

Textspalten: 29
Numerische Spalten: 7

X shape: (12972, 8)
y shape: (12972,)

Train size: 10377
Test size: 2595
Train dist:
 Employment
employed                                                0.860075
independent contractor, freelancer, or self-employed    0.111978
student                                                 0.014166
not employed                                            0.013780
Name: proportion, dtype: float64
Test dist:
 Employment
employed                                                0.860116
independent contractor, freelancer, or self-employed    0.111753
student                                                 0.014258
not employed             

In [6]:
# %% md
# XGBoost + OneHot-Encoding – JobSat (Low/Medium/High)
# Ziel: JobSat klassifizieren (3 Klassen)
# Features: Numerik + kategoriale Spalten via OneHotEncoder
# Modell: XGBClassifier

# %%
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix

from xgboost import XGBClassifier

# %%
# 1) Daten laden
df = pd.read_csv("survey_results_cleaned_final.csv")
print("Loaded:", df.shape)

# Optional: bool -> int (falls vorhanden)
bool_cols = df.select_dtypes(include=["bool"]).columns
for c in bool_cols:
    df[c] = df[c].astype(int)

# %%
# 2) Target: JobSat -> Klassen (Low/Medium/High)
target_col = "JobSat"

# Nur Zeilen behalten, wo JobSat vorhanden ist
df = df.dropna(subset=[target_col]).copy()

def map_jobsat(x):
    x = float(x)
    if x <= 3:
        return 0   # Low
    elif x <= 6:
        return 1   # Medium
    else:
        return 2   # High

y = df[target_col].apply(map_jobsat).astype(int)

print("Target distribution:")
print(y.value_counts().sort_index())

# %%
# 3) Feature-Spalten bestimmen
# -> Numerik: int/float (ohne JobSat, ohne cluster falls vorhanden)
# -> Kategorial: object (Strings), ebenfalls ohne cluster

drop_cols = {target_col}
if "cluster" in df.columns:
    drop_cols.add("cluster")

# Numerische Spalten
num_cols = df.select_dtypes(include=["int64","float64","int32","float32"]).columns.tolist()
num_cols = [c for c in num_cols if c not in drop_cols]

# Kategoriale Spalten
cat_cols = df.select_dtypes(include=["object"]).columns.tolist()
cat_cols = [c for c in cat_cols if c not in drop_cols]

# Optional: ID-Spalten raus (falls vorhanden)
for maybe_id in ["ResponseId"]:
    if maybe_id in num_cols: num_cols.remove(maybe_id)
    if maybe_id in cat_cols: cat_cols.remove(maybe_id)

print("Numeric cols:", len(num_cols))
print("Categorical cols:", len(cat_cols))

# %%
# 4) Gemeinsamer Clean-Step: nur Reihen behalten, wo Features + Target vollständig sind
# Für OneHot ist NaN ok (Encoder kann mit fehlenden umgehen, wenn wir sie als Kategorie behandeln),
# ABER XGBoost + sklearn Pipeline ist am stabilsten, wenn wir NaNs in cat als "MISSING" füllen.
X = df[num_cols + cat_cols].copy()

# Kategoriale NaNs füllen
for c in cat_cols:
    X[c] = X[c].fillna("MISSING").astype(str)

# Numerische NaNs: XGBoost kann NaNs grundsätzlich, aber damit train/test konsistent ist,
# lassen wir sie drin. Wenn du willst, kannst du hier auch dropna machen:
# X = X.dropna(subset=num_cols)
# y = y.loc[X.index]

# Wichtig: Index syncen
y = y.loc[X.index]
assert len(X) == len(y), f"Mismatch X={len(X)} vs y={len(y)}"

print("X shape:", X.shape, "| y shape:", y.shape)

# %%
# 5) Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))

# %%
# 6) Preprocessing: OneHotEncoder für Kategorien
#    Wichtig: seltene Kategorien bündeln -> verhindert Feature-Explosion.
#    min_frequency= z.B. 50 heißt: Kategorien die <50 mal vorkommen, werden "infrequent".
#    (Wenn sklearn zu alt ist und min_frequency nicht unterstützt -> sag Bescheid, ich gebe Fallback.)
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(
            handle_unknown="ignore",
            min_frequency=50,               # <<< ggf. anpassen (25 / 50 / 100)
            sparse_output=True
        ), cat_cols),
        ("num", "passthrough", num_cols),
    ],
    remainder="drop"
)

# %%
# 7) Modell: XGBoost Multiclass
xgb = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    tree_method="hist",
    n_jobs=-1,
    random_state=42
)

model = Pipeline([
    ("preprocess", preprocessor),
    ("clf", xgb)
])

# %%
# 8) Trainieren
model.fit(X_train, y_train)

# %%
# 9) Evaluation
y_pred = model.predict(X_test)

print("Classification Report (Test):")
print(classification_report(y_test, y_pred, labels=[0,1,2], target_names=["Low","Medium","High"]))

print("Confusion Matrix (rows=true, cols=pred):")
print(confusion_matrix(y_test, y_pred, labels=[0,1,2]))

# Optional: Train-Report (Overfitting-Check)
y_pred_train = model.predict(X_train)
print("\nClassification Report (Train):")
print(classification_report(y_train, y_pred_train, labels=[0,1,2], target_names=["Low","Medium","High"]))


Loaded: (19826, 38)
Target distribution:
JobSat
0     1022
1     3842
2    12351
Name: count, dtype: int64
Numeric cols: 6
Categorical cols: 30
X shape: (17215, 36) | y shape: (17215,)
Train size: 13772
Test size: 3443
Classification Report (Test):
              precision    recall  f1-score   support

         Low       0.33      0.06      0.11       204
      Medium       0.38      0.12      0.18       769
        High       0.75      0.96      0.84      2470

    accuracy                           0.72      3443
   macro avg       0.49      0.38      0.38      3443
weighted avg       0.64      0.72      0.65      3443

Confusion Matrix (rows=true, cols=pred):
[[  13   50  141]
 [  18   93  658]
 [   8  103 2359]]

Classification Report (Train):
              precision    recall  f1-score   support

         Low       0.99      0.41      0.58       818
      Medium       0.92      0.38      0.54      3073
        High       0.81      0.99      0.89      9881

    accuracy            

# Mindestanforderungen Klassifikation
- Führen Sie mit dem Algorithmus Ihrer Wahl eine Klassifikationsaufgabe auf Ihren Daten durch.
    - Ziel war die Vorhersage der Zielvariable `Employment`
    - Klassen waren _employed_, _self-employed_, _student_ und _not-employed_
    - sehr seltene Klassen wurden entfernt
    - Wir haben die **lineare Support Vector Machine** genutzt
    - Einsatz in der Pipeline mit Text- und numerischen Features
___
- Teilen Sie dazu zunächst die Daten auf, um Overfitting beim Trainieren des Algorithmus und bei der Parameterauswahl zu vermeiden. Erklären Sie die gewählte Strategie und die Größenverhältnisse.
    - Wir haben auf 80/20 gesplittet, also 80% Train und 20% Test (`train_test_split(test_size=0.2)`)
    - wir haben `stratify=y` angewandt um eine ähnliche Klassenverteilung in den Train- und Testdaten zu erhalten
    - um Overfitting bei Parametern zu verhindern haben wir Parameter-Tuning nur auf das Trainingsset mit 3-facher Cross-Validation angewandt
---
- Wählen Sie geeignete Features aus und setzen Sie die Parameter des Algorithmus. Beschreiben Sie das gewälhte Vorgehen für die Auswahl der Features und Parameter. Berichten Sie den Parameterraum und die final gewählten Parameter. Geben Sie die Performanz auf den Trainingsdaten (bzw. Entwicklungsdaten, falls verwendet) an.
    - **Features**:
        - alle Textspalten zu `__text__` zusammengefasst, TfidfVectorizer
        - bei numerischen Spalten wurden fehlende Werte mit dem Median aufgefüllt und anschließend Standardisiert
        - Entfernt wurden: `ResponseId`, `Age` (als String) und `ConvertedCompTotal` (Gehälter)
    - **Parameterauswahl**
        - GridSearchCV (cv=3) auf Trainingsdaten
        - Optimierungsmaß: macro-F1
    - **Parameterraum**:
        - `ngram_range`: (1,1), (1,2)
        - `min_df`: 5, 10
        - `max_df`: 0.9
        - `C`: 0.5, 1.0, 2.0
        - `class_weight`: None, balanced
    - **Beste Parameter**:
        - `ngram_range` = (1,2)
        - `min_df` = 10
        - `C` = 0.5
        - `class_weight` = balanced
    - **Train/Dev-Leistung**
        - CV macro-F1 ~ 0.75
---
- Evaluieren Sie die Klassifikation auf den ungesehenen Testdaten. Betrachten Sie Precision und Recall sowie den F-Wert. Welches Maß ist für Ihre Anwendung wichtiger? Bewerten Sie Ihr Ergebnis. Ist es in der Praxis voraussichtlich zufriedenstellend?
    - Test-Ergebnisse
        - Accuracy: 0.93
        - macro-F1: 0.77
    - Precision/Recall
        - sehr hoch für _employed_
        - gut für _self-employed_
        - geringer für _student_ und not _employed_ (kleine Klassen)
    - wichtigstes Maß
        - macro-F1 ist wichtiger als die Accuracy, da die Klassen sehr stark unbalanciert sind
    - Bewertung
        - das Modell erkennt Mehrheitsklassen sehr zuverlässig
        - Minderheitsklassen werden schwieriger erkannt, hier wären in der Praxis voraussichtlich weitere Maßnahmen nötig

# Codeerklärungen
- imports laden
- csv einlesen
- Zielvariable `Employment` festlegen, die durch Klassifikation vorhergesagt werden soll

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix

df = pd.read_csv("survey_results_cleaned_final.csv")
target_col = "Employment"

- Zeilen ohne `Employment` werden entfernt
- sehr seltene Klassen werden entfernt, da sie irrelevant sind und zu wenige Beispiele haben (`prefer not to say` & `Other`
- zudem wird `ResponseId` entfernt, da es nur eine ID ist - kein inhaltlicher Mehrwert
- `Age` ist redundant aufgrund von `AgeNum`
- `ConvertedCompTotal` zu viele leere Zellen/NaN-Werte

In [ ]:
df = df.dropna(subset=[target_col]).copy()
df = df[df[target_col] != "i prefer not to say"].copy()
df = df[df[target_col] != "Other"].copy()

df = df.drop(columns=["ResponseId", "Age", "ConvertedCompTotal"], errors="ignore")

- Nach `object`-Spalten suchen, da sie meist Text/Kategorien sind
- `Employment` muss entfernt werden, da sonst geschummelt werden würde
- Aus allen Textspalten eine gemeinsamen Text pro Zeile (`__text__`)
#### Warum?
- kategoriale Spalten als Textklassifikation zu behandeln
- Danach kann TFIDF Vectorizer wie beim Clustering numerische Features daraus machen

In [ ]:
text_cols = df.select_dtypes(include=["object"]).columns.tolist()
text_cols = [c for c in text_cols if c not in {target_col, "cluster"}]
df["__text__"] = df[text_cols].fillna("").agg(" ".join, axis=1)

- numerische Spalten sammeln
- Auch hier wird Zielvariable `Employment` ausgeschlossen

In [ ]:
num_cols = df.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()
num_cols = [c for c in num_cols if c not in {target_col, "cluster"}]

- `x` enthält Features (Text und Numerisch)
- `y` enthält die Labels, also die Employment-Klassen
- Ausgabe der Shapes und Klassenverteilung um Werte zu überprüfen

In [ ]:
X = df[["__text__"] + num_cols].copy()
y = df[target_col].astype(str).copy()

print("X shape:", X.shape, "| y shape:", y.shape)
print("Target distribution:\n", y.value_counts())

- wir splitten in 80% Training und 20% Test
- `stratify=y` -> Klassenverteilung bleibt bei Train und Test ungefähr gleich
- stratify wichtig, weil Zielvariable stark unbalanciert ist

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

### Textspalten
- `TfidfVectorizer` macht aus Wörtern Zahlen
- dabei werden Wörter so gewichtet, dass häufige "Standardwörter" weiger wichtig sind und informative Wörter stärker zählen
- `max_features=20000` begrenzt die Anzahl der Textfeatures, damit die Werte nicht explodieren
- `sublinear_tf=True` dämpft extrem häufige Wörte zusätzlich

### Numerische Spalten
- `SimpleImputer(median)` füllt fehlende numerische Werte, also `NaN-Werte`, mit dem Median der Spalte
- StandardScaler skaliert die Zahlen auf vergleichbare Größenordnungen, was Support Vector Machines hilft, da sie empfindlich auf unterschiedliche Skalen reagieren können

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("text", TfidfVectorizer(sublinear_tf=True, max_features=20000), "__text__"),
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), num_cols)
    ],
    remainder="drop"
)

- in der Pipeline kommt nun die Vorverarbeitung und der Klassifikatior (LinearSVC) zusammen
- Warum LinearSVC?
    - für Textdaten mit vielen Features funktioniert lineare SVC oft sehr gut

In [ ]:
pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", LinearSVC(max_iter=20000))
])

- hier werden mehrere sinnvolle Einstellungen getestet:
- `ngram_range`:
    - (1,1) = nur einzelne Wörter
    - (1,2) = Wörter + Wortpaare -> Kontext besser zu erfassen
- `min_df`:
    - ignoriert sehr seltene Wörter (erzeugt oft nur rauschen)
- `C`:
    - Regularisierung der SVM: kleiner C entspricht stärkerer Regularisierung, ein größeres C bedeutet mehr Flexibilität
- `class_weight="balanced"`:
    - wichtig bei unbalancierten Klassen, da so kleine Klassen stärker gewichtet werden

In [ ]:
param_grid = {
    "preprocessing__text__ngram_range": [(1, 1), (1, 2)],
    "preprocessing__text__min_df": [5, 10],
    "preprocessing__text__max_df": [0.9],
    "classifier__C": [0.5, 1.0, 2.0],
    "classifier__class_weight": [None, "balanced"],
}

- GridSearchCV testet alle Parameterkombinationen
- Bewertung: `f1_macro`
    - sinnvoll, weil jede Klasse gleich gewichtet wird
- cv=3 bedeutet 3-fache Cross-Validation auf dem Trainingsset
- Testset bleibt unangetastet -> fairer Vergleich

In [ ]:
grid = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=3,
    n_jobs=-1,
    verbose=2
)

grid.fit(X_train, y_train)

print("\nBest CV macro-F1:", grid.best_score_)
print("Best params:", grid.best_params_)

- bestes Modell aus der GridSearch wird verwendet
- dann einmalige Evaluierung auf den Testdaten
- Confusion Matrix zeigt, welche Klassen miteinander verwechselt wurden
- Mehrheitsklasse (employed) oft sehr gut
- Minderheitsklassen häufiger als employed vorhergesagt

In [5]:
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

print("\nTest report:\n", classification_report(y_test, y_pred, zero_division=0))

labels_sorted = best_model.named_steps["classifier"].classes_
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred, labels=labels_sorted))

X shape: (19810, 7) | y shape: (19810,)
Target distribution:
 Employment
employed                                                16316
independent contractor, freelancer, or self-employed     2483
student                                                   659
not employed                                              352
Name: count, dtype: int64
Fitting 3 folds for each of 24 candidates, totalling 72 fits

Best CV macro-F1: 0.7450045659917125
Best params: {'classifier__C': 0.5, 'classifier__class_weight': 'balanced', 'preprocessing__text__max_df': 0.9, 'preprocessing__text__min_df': 10, 'preprocessing__text__ngram_range': (1, 2)}

Test report:
                                                       precision    recall  f1-score   support

                                            employed       0.96      0.98      0.97      3263
independent contractor, freelancer, or self-employed       0.85      0.77      0.81       497
                                        not employed       0.71  